# Pixel-level confusion matrix on the held-out test set

Runs the current best UNet over the **held-out InitialLabels test split** (the 2 images not in val — `D16_03_1_15` and `D16_03_1_1`) and produces a 4×4 pixel-level confusion matrix.

Two views are rendered side-by-side:

- **Row-normalized** (recall view): each row sums to 100 %; the diagonal is per-class recall.
- **Column-normalized** (precision view): each column sums to 100 %; the diagonal is per-class precision.

Classes: 0 = polymer, 1 = background, 2 = cell body, 3 = cell boundary.

Used for the paper figure showing UNet's pixel-classification performance (cell body and boundary in particular). The protrusion length test goes through `protrusion_length_grid.ipynb`.

In [ ]:
import sys, os
from pathlib import Path

# Bootstrap sys.path so we can import polymer_detection. When the notebook
# lives at sickling/notebooks/, cwd = sickling/notebooks/ and the parent
# (sickling/) holds the protrusion_detection package (formerly polymer_detection).
_BOOTSTRAP = Path(os.path.abspath('')).parent
if str(_BOOTSTRAP) not in sys.path:
    sys.path.insert(0, str(_BOOTSTRAP))

# Now the canonical anchor — the package's own location — to confirm the
# correct REPO_ROOT regardless of where the notebook was launched from.
import polymer_detection as _pd
REPO_ROOT = Path(_pd.__file__).resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image

from sickling.protrusion_detection.config import cfg
from sickling.protrusion_detection.inference import load_unet, predict_probs, predict_probs_tta
from sickling.protrusion_detection.masks import load_bootstrap_label, normalize_image
from sickling.protrusion_detection.metrics import confusion_matrix
from sickling.protrusion_detection.splits import build_val_test_pairs

# Keep text editable in the SVG so the paper figure can be touched up in Illustrator/Inkscape.
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['pdf.fonttype'] = 42

print('REPO_ROOT      =', REPO_ROOT)
print('cfg.MODELS_DIR =', cfg.MODELS_DIR)
print('device         =', cfg.DEVICE)

In [ ]:
# --- pick the models + test set ------------------------------------------
# Compares the **pre-HITL clean baseline** (loop_0 = fold 2, the operator's
# backup model that the densify path also uses) against the **current best
# post-HITL model** (loop_5 = fold 1, the carryover winner). Shows the
# pixel-level confusion matrix moving from before to after the iterative
# correction loop ran.
CKPTS = [
    ('loop_0 (pre-HITL baseline)',  'unet_fold_2_best_loop_0.pth'),
    ('loop_5 (after HITL)',         'unet_fold_1_best_loop_5.pth'),
]
USE_TTA      = True

CLASS_NAMES  = ['polymer', 'background', 'cell body', 'cell boundary']
COLORMAP     = 'Blues'

FIGURES_DIR  = REPO_ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Mirror the production val/test split so the 'held-out test' here really
# is the same set excluded from training. (D16_03_1_15 and D16_03_1_1.)
cfg.VAL_STEMS = [
    'D16_03_1_12_Bright Field_001',
    'D16_03_1_13_Bright Field_001',
    'D16_03_1_17_Bright Field_001',
]
val_pairs, test_pairs = build_val_test_pairs()

print(f'val  ({len(val_pairs)}) : ' + ', '.join(os.path.splitext(os.path.basename(p[0]))[0] for p in val_pairs))
print(f'test ({len(test_pairs)}) : ' + ', '.join(os.path.splitext(os.path.basename(p[0]))[0] for p in test_pairs))
for label, ckpt in CKPTS:
    print(f'ckpt  ({label:25s}): {ckpt}  (TTA={USE_TTA})')
print(f'output dir    : {FIGURES_DIR}')

In [ ]:
# Run inference per checkpoint, accumulate the pixel confusion matrix.
N = cfg.N_CLASSES
predict = predict_probs_tta if USE_TTA else predict_probs

results = {}                 # label -> {'cm': ndarray, 'per_image': [...]}
for label, ckpt_name in CKPTS:
    ckpt_path = os.path.join(cfg.MODELS_DIR, ckpt_name)
    print(f'\\n=== {label}  ({ckpt_name}) ===')
    model = load_unet(ckpt_path, device=cfg.DEVICE).eval()

    total_cm = np.zeros((N, N), dtype=np.int64)
    per_image = []
    for raw_path, mask_path in test_pairs:
        stem = os.path.splitext(os.path.basename(raw_path))[0]
        img = np.array(Image.open(raw_path).convert('L'), dtype=np.float32)
        t = torch.from_numpy(normalize_image(img)).float().unsqueeze(0).to(cfg.DEVICE)
        with torch.no_grad():
            probs = predict(model, t)
        pred = torch.argmax(probs, dim=0)
        gt   = torch.from_numpy(load_bootstrap_label(mask_path).astype(np.int64)).to(pred.device)
        cm = confusion_matrix(pred, gt)
        total_cm += cm
        row_sums = cm.sum(axis=1)
        per_class_recall = [float(cm[c, c] / row_sums[c]) if row_sums[c] > 0 else float('nan')
                            for c in range(N)]
        per_image.append((stem, per_class_recall, int(cm.sum())))
    del model
    results[label] = {'cm': total_cm, 'per_image': per_image, 'ckpt': ckpt_name}

    print(f'aggregate confusion matrix (counts):')
    print(total_cm)
    print(f'\\ntotal pixels graded: {int(total_cm.sum()):,}')
    print('per-image, per-class recall:')
    for stem, rec, n in per_image:
        rec_str = '  '.join(f'{CLASS_NAMES[i]}={rec[i]:.3f}' for i in range(N))
        print(f'  {stem[:48]:48s}  n_px={n:>10,}  {rec_str}')

In [ ]:
# Render side-by-side comparison. 2 rows (one per checkpoint), 2 cols
# (recall view, precision view). The visual delta between top row and
# bottom row IS the per-class effect of the HITL correction loop.

def _draw_heatmap(ax, matrix, title, cbar_label, classes, cmap='Blues'):
    im = ax.imshow(matrix, cmap=cmap, vmin=0, vmax=100, aspect='equal')
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=30, ha='right')
    ax.set_yticklabels(classes)
    ax.set_xlabel('predicted class')
    ax.set_ylabel('ground-truth class')
    ax.set_title(title)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            v = matrix[i, j]
            color = 'white' if v > 55 else 'black'
            label = f'{v:.1f}%' if v >= 0.1 else (f'{v:.2f}%' if v > 0 else '0%')
            ax.text(j, i, label, ha='center', va='center', fontsize=9, color=color)
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label(cbar_label)


fig, axes = plt.subplots(len(CKPTS), 2, figsize=(13, 5.4 * len(CKPTS)))
if len(CKPTS) == 1:
    axes = axes.reshape(1, -1)

for r, (label, _) in enumerate(CKPTS):
    cm = results[label]['cm']
    row_sums = cm.sum(axis=1, keepdims=True)
    recall_view = np.divide(cm, row_sums,
                            out=np.zeros_like(cm, dtype=float),
                            where=row_sums > 0) * 100.0
    col_sums = cm.sum(axis=0, keepdims=True)
    precision_view = np.divide(cm, col_sums,
                               out=np.zeros_like(cm, dtype=float),
                               where=col_sums > 0) * 100.0
    _draw_heatmap(axes[r, 0], recall_view,
                  f'{label}\nRow-normalized (recall view)',
                  '% of true class', CLASS_NAMES, cmap=COLORMAP)
    _draw_heatmap(axes[r, 1], precision_view,
                  f'{label}\nColumn-normalized (precision view)',
                  '% of predicted class', CLASS_NAMES, cmap=COLORMAP)

total_pixels = int(next(iter(results.values()))['cm'].sum())
fig.suptitle(
    f'UNet pixel-level confusion matrix on held-out test '
    f'({len(test_pairs)} images, {total_pixels:,} pixels)',
    y=1.005,
)
fig.tight_layout()

svg_path = FIGURES_DIR / 'pixel_confusion_matrix.svg'
png_path = FIGURES_DIR / 'pixel_confusion_matrix.png'
fig.savefig(svg_path, bbox_inches='tight')
fig.savefig(png_path, dpi=200, bbox_inches='tight')
plt.show()

print(f'wrote: {svg_path}')
print(f'wrote: {png_path}')

In [ ]:
import csv

# Per-checkpoint counts CSV (one file per ckpt).
for label, payload in results.items():
    cm = payload['cm']
    tag = payload['ckpt'].replace('.pth', '')
    csv_path = FIGURES_DIR / f'pixel_confusion_counts_{tag}.csv'
    with open(csv_path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow([''] + [f'predicted_{c}' for c in CLASS_NAMES])
        for i, c in enumerate(CLASS_NAMES):
            w.writerow([f'truth_{c}'] + [int(cm[i, j]) for j in range(len(CLASS_NAMES))])
    print(f'wrote: {csv_path}')

# Per-class recall / precision / F1 side-by-side per checkpoint.
print()
header = f'{"class":15s}'
for label, _ in CKPTS:
    header += f'  | {label:25s} (R / P / F1)'
print(header)

summary_rows = []
for c in range(N):
    line = f'{CLASS_NAMES[c]:15s}'
    row = {'class': CLASS_NAMES[c]}
    for label, _ in CKPTS:
        cm = results[label]['cm']
        tp = int(cm[c, c])
        fn = int(cm[c].sum() - tp)
        fp = int(cm[:, c].sum() - tp)
        r  = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
        p  = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
        f1 = 2 * r * p / (r + p) if (r + p) > 0 and not (np.isnan(r) or np.isnan(p)) else float('nan')
        line += f'  | {r:5.3f} / {p:5.3f} / {f1:5.3f}              '
        row[f'{label}__recall']    = r
        row[f'{label}__precision'] = p
        row[f'{label}__f1']        = f1
    summary_rows.append(row)
    print(line)

# Side-by-side summary CSV (for the paper text).
summary_csv = FIGURES_DIR / 'pixel_confusion_summary.csv'
import pandas as pd
pd.DataFrame(summary_rows).to_csv(summary_csv, index=False)
print(f'\nwrote: {summary_csv}')